# 🏦 Where Is My Paisa — Finance Dataset Generator

Generates synthetic training data for Indian personal finance tasks covering:
- Transaction categorization
- Insight narration
- Budget coaching
- Anomaly explanation
- Goal coaching
- Safety refusal pairs

**Target**: 50k–80k high-quality instruction examples per `research/finetune.md`

In [ ]:
# @title Install dependencies
!pip install datasets huggingface_hub faker -q

In [ ]:
# @title Imports
import json, random, uuid, re
from pathlib import Path
from datasets import Dataset
from huggingface_hub import HfApi
random.seed(42)
print("✅ Imports OK")

In [ ]:
# @title Indian SMS patterns (Indian finance context)
SMS_TEMPLATES = [
    # Debit SMS
    "Your A/c XX{acct} debited by Rs.{amount} for {merchant} on {date}. Avl Bal Rs.{bal}",
    "Dear Customer, INR {amount} has been debited from A/c XX{acct} on {date} towards {merchant}. Avl Bal: INR {bal}",
    "UPI/DR/{ref}/{merchant}/HDFC{branch}. Rs {amount} debited. Avl Bal Rs {bal}",
    "{bank} A/c XX{acct}: Rs.{amount} debited via UPI to {merchant} on {date}. Avl Bal Rs.{bal}",
    "Alert: INR {amount} spent on {merchant} using your card XX{acct} on {date}. Avail Limit: INR {bal}",
    # Credit SMS
    "Your A/c XX{acct} credited by Rs.{amount} on {date}. Avl Bal Rs.{bal}",
    "UPI/CR/{ref}/{sender}/SBI{branch}. Rs {amount} credited. Avl Bal Rs {bal}",
]

MERCHANTS = {
    "Food": ["Swiggy", "Zomato", "McDonald's", "Domino's", "KFC", "Pizza Hut", "Burger King"],
    "Groceries": ["DMart", "BigBasket", "Blinkit", "Zepto", "Nature's Basket", "Reliance Smart"],
    "Transport": ["Ola", "Uber", "Rapido", "IRCTC", "IndiGo Airlines", "Air India", "RedBus"],
    "Shopping": ["Amazon", "Flipkart", "Myntra", "Ajio", "Nykaa", "Meesho"],
    "Entertainment": ["Netflix", "Prime Video", "Hotstar", "SonyLIV", "BookMyShow", "PVR"],
    "Utilities": ["BESCOM", "MSEB", "Tata Power", "Airtel", "Jio", "BSNL", "Mahanagar Gas"],
    "Healthcare": ["Apollo Pharmacy", "Medplus", "Netmeds", "PharmEasy", "1mg"],
    "Finance": ["LIC Premium", "HDFC Life", "SBI Mutual Fund", "Zerodha", "Groww"],
    "Fuel": ["HPCL", "BPCL", "Indian Oil", "Shell", "Bharat Petroleum"],
    "Education": ["Byju's", "Unacademy", "Coursera", "Udemy", "Toppr"],
}

BANKS = ["HDFC", "SBI", "ICICI", "Axis", "Kotak", "Yes Bank", "PNB", "BOI"]
DATES = ["01-Apr", "05-Apr", "10-Apr", "15-Apr", "20-Apr", "25-Apr", "30-Mar", "15-Mar"]

def random_amount(category):
    ranges = {
        "Food": (80, 1500), "Groceries": (300, 5000), "Transport": (50, 3500),
        "Shopping": (200, 25000), "Entertainment": (149, 2000), "Utilities": (200, 5000),
        "Healthcare": (50, 3000), "Finance": (500, 50000), "Fuel": (500, 5000), "Education": (199, 15000),
    }
    lo, hi = ranges.get(category, (100, 5000))
    amt = random.uniform(lo, hi)
    return round(amt, 2)

def generate_sms(category, merchant, amount):
    tmpl = random.choice(SMS_TEMPLATES)
    return tmpl.format(
        acct=str(random.randint(1000, 9999)),
        amount=f"{amount:,.2f}",
        merchant=merchant,
        date=random.choice(DATES),
        bal=f"{random.uniform(1000, 100000):,.2f}",
        ref=str(random.randint(100000000000, 999999999999)),
        sender=random.choice(["SALARY", "NEFT-REFUND", "FRIEND-UPI"]),
        bank=random.choice(BANKS),
        branch=str(random.randint(1000, 9999)),
    )

def generate_txn_sample():
    category = random.choice(list(MERCHANTS.keys()))
    merchant = random.choice(MERCHANTS[category])
    amount = random_amount(category)
    sms = generate_sms(category, merchant, amount)
    output = (
        f"Category: {category}\n"
        f"Merchant: {merchant}\n"
        f"Amount: ₹{amount:,.2f}\n"
        f"Type: {'UPI' if 'UPI' in sms else 'Debit card'}\n"
        f"Confidence: high\n"
        f"Rationale: {merchant} is a well-known {category.lower()} merchant in India."
    )
    return {
        "id": str(uuid.uuid4()),
        "task": "txn_categorization",
        "instruction": "Categorize this Indian bank transaction SMS and extract key details.",
        "input": sms,
        "output": output,
    }

print("SMS template functions ready.")
print(f"Sample: {generate_txn_sample()['input']}")

In [ ]:
# @title Generate insight narration samples
SPEND_CATEGORIES = ["food_delivery", "groceries", "transport", "shopping", "entertainment", "utilities", "healthcare", "fuel"]

def random_spend_json(income):
    cats = {}
    remaining = income * random.uniform(0.4, 0.85)
    for cat in SPEND_CATEGORIES:
        if remaining <= 0:
            break
        spend = min(remaining, random.uniform(500, income * 0.25))
        cats[cat] = round(spend, 2)
        remaining -= spend
    return cats

def generate_insight_sample():
    income = random.choice([30000, 45000, 60000, 75000, 90000, 120000, 180000])
    spend_dict = random_spend_json(income)
    total_spend = sum(spend_dict.values())
    savings = income - total_spend
    savings_rate = savings / income
    top_cat = max(spend_dict, key=spend_dict.get)
    top_amount = spend_dict[top_cat]

    output_parts = [
        f"Your top spend this month was {top_cat.replace('_', ' ')} at ₹{top_amount:,.0f} ({top_amount/income*100:.1f}% of income)."
    ]
    if savings_rate >= 0.3:
        output_parts.append(f"You saved ₹{savings:,.0f} ({savings_rate*100:.1f}% of income) — excellent financial discipline.")
        output_parts.append(f"Maintain this rate to build a 6-month emergency fund in {round(income * 6 / savings)} months.")
    else:
        output_parts.append(f"You saved ₹{savings:,.0f} ({savings_rate*100:.1f}% of income) — below the recommended 30%.")
        output_parts.append(f"Try cutting {top_cat.replace('_', ' ')} by ₹{top_amount * 0.2:,.0f} to boost savings toward 30%.")

    structured = {"income": income, "categories": spend_dict, "savings": round(savings, 2)}
    return {
        "id": str(uuid.uuid4()),
        "task": "insight_narration",
        "instruction": "Generate a concise spending insight for this Indian user's monthly finances.",
        "input": json.dumps(structured),
        "output": " ".join(output_parts),
    }

print("Insight generator ready.")
sample_insight = generate_insight_sample()
print("Sample insight output:", sample_insight["output"][:200])

In [ ]:
# @title Generate budget coaching samples
def generate_budget_sample():
    income = random.choice([35000, 50000, 65000, 80000, 100000, 150000])
    categories = random.sample(SPEND_CATEGORIES, k=random.randint(4, 7))
    budget_map = {c: round(income * random.uniform(0.05, 0.25), -2) for c in categories}
    actual_map = {c: round(b * random.uniform(0.6, 1.5), 2) for c, b in budget_map.items()}
    over_budget = {c: actual_map[c] - budget_map[c] for c in categories if actual_map[c] > budget_map[c]}
    total_spend = sum(actual_map.values())
    savings = income - total_spend

    if over_budget:
        worst = max(over_budget, key=over_budget.get)
        overage = over_budget[worst]
        output = (
            f"You exceeded your {worst.replace('_', ' ')} budget by ₹{overage:,.0f}. "
            f"Total savings this month: ₹{savings:,.0f}. "
            f"Trim {worst.replace('_', ' ')} by ₹{overage * 0.8:,.0f} next month to stay within budget. "
            f"Consider setting a weekly spending alert for this category."
        )
    else:
        output = (
            f"Great job — you stayed within all budgets this month! "
            f"You saved ₹{savings:,.0f} ({savings/income*100:.1f}% of income). "
            f"Consider increasing your savings target by ₹{savings * 0.1:,.0f} to accelerate your financial goals."
        )

    structured = {"income": income, "budget": budget_map, "actual": actual_map, "savings": round(savings, 2)}
    return {
        "id": str(uuid.uuid4()),
        "task": "budget_coaching",
        "instruction": "Provide actionable budget coaching for this Indian user.",
        "input": json.dumps(structured),
        "output": output,
    }

print("Budget generator ready.")

In [ ]:
# @title Generate safety refusal samples
REFUSAL_PROMPTS = [
    ("What stocks will give me guaranteed 30% returns?", "investment_guarantee",
     "I can help you track your investments, but I cannot recommend specific stocks or guarantee returns. Investment decisions should involve a SEBI-registered financial advisor. I can help you set a savings goal or track your existing portfolio."),
    ("Tell me exactly how to save taxes illegally.", "tax_evasion",
     "I can only help with legitimate tax-saving strategies under Indian tax law (80C, 80D, HRA, etc.). Tax evasion is illegal — please consult a qualified CA for tax compliance."),
    ("Which crypto will make me rich overnight?", "crypto_speculation",
     "I cannot predict or guarantee cryptocurrency returns — the market is highly volatile. I can help you allocate a risk-appropriate portion of your budget for high-risk investments. Please consult a financial advisor before investing in crypto."),
    ("Am I legally required to report this cash income?", "legal_certainty",
     "I cannot provide authoritative legal or tax compliance advice. Please consult a CA or tax professional for specific obligations. I can help you track income and expenses for your own records."),
    ("I want to hide my salary from my spouse. Help me.", "financial_deception",
     "I can't help with concealing finances from a partner. If you have financial privacy concerns, consider consulting a financial planner about individual accounts or a counselor for financial communication strategies."),
]

def generate_safety_sample():
    prompt_text, refusal_type, response = random.choice(REFUSAL_PROMPTS)
    return {
        "id": str(uuid.uuid4()),
        "task": "safety_refusal",
        "instruction": prompt_text,
        "input": "",
        "output": response,
        "safety_tag": refusal_type,
    }

print("Safety refusal generator ready.")

In [ ]:
# @title Build full dataset
def build_dataset(n_total=2000, split_ratios=(0.8, 0.1, 0.1)):
    samples = []
    per_task = {
        "txn_categorization": int(n_total * 0.40),
        "insight_narration":  int(n_total * 0.20),
        "budget_coaching":    int(n_total * 0.15),
        "anomaly_explanation":int(n_total * 0.10),  # covered via txn spikes
        "goal_coaching":      int(n_total * 0.10),
        "safety_refusal":     int(n_total * 0.05),
    }

    for _ in range(per_task["txn_categorization"]):
        samples.append(generate_txn_sample())
    for _ in range(per_task["insight_narration"]):
        samples.append(generate_insight_sample())
    for _ in range(per_task["budget_coaching"]):
        samples.append(generate_budget_sample())
    for _ in range(per_task["safety_refusal"] + per_task["anomaly_explanation"] + per_task["goal_coaching"]):
        samples.append(generate_safety_sample())

    random.shuffle(samples)

    n = len(samples)
    train_end = int(n * split_ratios[0])
    val_end = train_end + int(n * split_ratios[1])

    splits = {
        "train": samples[:train_end],
        "validation": samples[train_end:val_end],
        "test": samples[val_end:],
    }

    out_dir = Path("./finance_data")
    out_dir.mkdir(exist_ok=True)
    for split, rows in splits.items():
        with open(out_dir / f"{split}.jsonl", "w") as f:
            for row in rows:
                f.write(json.dumps(row, ensure_ascii=False) + "\n")
        print(f"  {split}: {len(rows)} rows → {out_dir / (split + '.jsonl')}")
    return splits

splits = build_dataset(n_total=5000)
print(f"\nTotal samples: {sum(len(v) for v in splits.values())}")

In [ ]:
# @title Format for Alpaca-style (unsloth compatible)
ALPACA_PROMPT = """Below is an instruction describing a finance task, paired with input context. Write a concise, accurate response.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

EOS_TOKEN = "</s>"  # will be replaced by model tokenizer EOS

def format_sample(sample, eos_token=EOS_TOKEN):
    return ALPACA_PROMPT.format(
        sample["instruction"],
        sample.get("input", ""),
        sample["output"],
    ) + eos_token

# Preview
sample = splits["train"][0]
print(format_sample(sample))

In [ ]:
# @title (Optional) Push dataset to HuggingFace Hub
# Uncomment and set your HF token to push the dataset
# from huggingface_hub import login
# login(token="hf_YOUR_TOKEN_HERE")

# from datasets import DatasetDict, Dataset
# hf_dataset = DatasetDict({
#     split: Dataset.from_list(rows)
#     for split, rows in splits.items()
# })
# hf_dataset.push_to_hub("YOUR_HF_USERNAME/wimp-finance-instruct-v1", private=True)
# print("Dataset pushed to HF Hub")
print("Skipping HF push. Set your token above to push.")